In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MultiLabelBinarizer
import re
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
from time import time

from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, roc_auc_score, confusion_matrix)


In [ ]:
df = pd.read_excel('ttc-bus-delay-data-2023.xlsx')

In [ ]:
df.head()

,Route,Time,Day,Location,Incident,Min Delay
0,91,02:30,Sunday,WOODBINE AND MORTIMER,Diversion,81
1,69,02:34,Sunday,WARDEN STATION,Security,22
2,35,03:06,Sunday,JANE STATION,Cleaning - Unsanitary,30
3,900,03:14,Sunday,KIPLING STATION,Security,17
4,85,03:43,Sunday,MEADOWALE LOOP,Security,1


In [ ]:
df.describe(include='all')

,Route,Time,Day,Location,Incident,Min Delay
count,55637.0,56207,56207,56207,56207,56207.000000
unique,264.0,1440,7,11684,13,NaN
top,32.0,16:00,Friday,KENNEDY STATION,Mechanical,NaN
freq,1920.0,113,9004,1231,19235,NaN
mean,NaN,NaN,NaN,NaN,NaN,20.251606
std,NaN,NaN,NaN,NaN,NaN,50.170167
min,NaN,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,NaN,9.000000
50%,NaN,NaN,NaN,NaN,NaN,11.000000
75%,NaN,NaN,NaN,NaN,NaN,20.000000


In [ ]:
df.dtypes

,0
Route,object
Time,object
Day,object
Location,object
Incident,object
Min Delay,int64


In [ ]:
# 1. Clean and standardize location data
def clean_location(loc):
    # Replace common intersection indicators with standardized ' AND '
    loc = re.sub(r'(&|\bat\b)', ' AND ', str(loc), flags=re.IGNORECASE)
    # Remove special characters (including dashes), numbers, and extra spaces
    loc = re.sub(r'[^a-zA-Z\s]', '', loc)  # Removes dashes and other special chars
    # Collapse multiple ANDs and spaces
    loc = re.sub(r'\s+AND\s+', ' AND ', loc, flags=re.IGNORECASE)
    loc = re.sub(r'\s+', ' ', loc).strip().upper()
    return loc

df['Location'] = df['Location'].apply(clean_location)

# 2. Split locations into components (handling AND, &, and at)
df['Location_Components'] = df['Location'].str.split(' AND ')

# 3. Calculate component frequencies
all_components = df['Location_Components'].explode()
component_counts = all_components.value_counts()

# 4. Get components with >20 occurrences
frequent_components = component_counts[component_counts > 20].index.tolist()

# 5. Filter rows where all components are frequent
mask = df['Location_Components'].apply(
    lambda x: all(comp in frequent_components for comp in x)
)
df_filtered = df[mask].copy()

# 6. Create binary encoding for frequent components
if frequent_components:
    mlb = MultiLabelBinarizer(classes=frequent_components)
    encoded_locations = mlb.fit_transform(df_filtered['Location_Components'])

    location_df = pd.DataFrame(encoded_locations,
                             columns=mlb.classes_,
                             index=df_filtered.index)
    df_encoded = pd.concat([df_filtered, location_df], axis=1)

    # Cleanup columns
    df_encoded = df_encoded.drop(columns=['Location_Components'])
else:
    print("Warning: No locations met the frequency threshold!")
    df_encoded = df_filtered

# 7. Reset index if needed
df_encoded = df_encoded.reset_index(drop=True)

# Show results
print(f"Original rows: {len(df)}")
print(f"Kept rows after filtering: {len(df_encoded)}")
print(f"Total location components found: {len(component_counts)}")
print(f"Frequent components (>20 occurrences): {len(frequent_components)}")
print("\nSample frequent components:", frequent_components[:10])

Original rows: 56207
Kept rows after filtering: 39149
Total location components found: 7943
Frequent components (>20 occurrences): 427

Sample frequent components: ['EGLINTON', 'FINCH', 'LAWRENCE', 'STEELES', 'JANE', 'SHEPPARD', 'DUFFERIN', 'BATHURST', 'YONGE', 'WESTON']


In [ ]:
df_encoded.head()

,Route,Time,Day,Location,Incident,Min Delay,EGLINTON,FINCH,LAWRENCE,STEELES,...,NUGGET,LUMSDEN,DUNDAS WEST,WESTHUMBER,WELLESL,INDUSTRY,WEATHERELL,MORNINGSID,AUKLAND,MURRAY ROSS
0,91,02:30,Sunday,WOODBINE AND MORTIMER,Diversion,81,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,69,02:34,Sunday,WARDEN STATION,Security,22,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,35,03:06,Sunday,JANE STATION,Cleaning - Unsanitary,30,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,900,03:14,Sunday,KIPLING STATION,Security,17,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,40,03:47,Sunday,KIPLING STATION,Emergency Services,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_encoded.isnull().sum()

,0
Route,319
Time,0
Day,0
Location,0
Incident,0
...,...
INDUSTRY,0
WEATHERELL,0
MORNINGSID,0
AUKLAND,0


In [ ]:
df_encoded.dropna(inplace=True)

In [ ]:
df_encoded.isnull().sum()

,0
Route,0
Time,0
Day,0
Location,0
Incident,0
...,...
INDUSTRY,0
WEATHERELL,0
MORNINGSID,0
AUKLAND,0


In [ ]:
df_encoded['Route'].value_counts()

,count
Route,
32,1465
36,1316
35,991
52,897
29,875
...,...
LINE 1 SHUTTLE - 600,1
CHANGEOF,1
1068,1


In [ ]:
# Robust text filtering
def is_numeric_route(route):
    try:
        # Handle NaN/None, strings, and numeric types
        return pd.notna(route) and str(route).isdigit()
    except:
        return False

# Apply filtering
mask = df_encoded['Route'].apply(is_numeric_route)
df_clean = df_encoded[mask].copy()

# Convert to integers (now safe)
df_clean['Route'] = df_clean['Route'].astype(int)

print(f"Original rows: {len(df_encoded)}")
print(f"Clean rows: {len(df_clean)}")
print("\nCleaned data:")
print(df_clean)

Original rows: 38830
Clean rows: 38691

Cleaned data:
       Route   Time     Day               Location               Incident  \
0         91  02:30  Sunday  WOODBINE AND MORTIMER              Diversion   
1         69  02:34  Sunday         WARDEN STATION               Security   
2         35  03:06  Sunday           JANE STATION  Cleaning - Unsanitary   
3        900  03:14  Sunday        KIPLING STATION               Security   
4         40  03:47  Sunday        KIPLING STATION     Emergency Services   
...      ...    ...     ...                    ...                    ...   
39144     41  01:26  Sunday     KEELE AND EGLINTON  Operations - Operator   
39145     94  01:31  Sunday   CASTLE FRANK STATION     Emergency Services   
39146     63  01:40  Sunday    OAKWOOD AND VAUGHAN              Diversion   
39147     34  01:54  Sunday       EGLINTON STATION     Emergency Services   
39148     41  01:55  Sunday       KEELE AND ROGERS          General Delay   

       Min Delay  EGL

In [ ]:
df_clean['Route'].value_counts()

,count
Route,
32,1465
36,1316
35,991
52,897
29,875
...,...
450,1
9525,1
500,1


In [ ]:
df_clean.drop(columns=['Location'],inplace=True)

In [ ]:
df_clean

,Route,Time,Day,Incident,Min Delay,EGLINTON,FINCH,LAWRENCE,STEELES,JANE,...,NUGGET,LUMSDEN,DUNDAS WEST,WESTHUMBER,WELLESL,INDUSTRY,WEATHERELL,MORNINGSID,AUKLAND,MURRAY ROSS
0,91,02:30,Sunday,Diversion,81,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,69,02:34,Sunday,Security,22,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,35,03:06,Sunday,Cleaning - Unsanitary,30,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,900,03:14,Sunday,Security,17,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,40,03:47,Sunday,Emergency Services,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39144,41,01:26,Sunday,Operations - Operator,28,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
39145,94,01:31,Sunday,Emergency Services,10,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
39146,63,01:40,Sunday,Diversion,33,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
39147,34,01:54,Sunday,Emergency Services,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Initialize TargetEncoder with smoothing to prevent overfitting
encoder = TargetEncoder(smooth="auto", target_type='continuous')

# Fit and transform the Route column
df_clean['Route_Encoded'] = encoder.fit_transform(df_clean[['Route']], df_clean['Min Delay'])

print("After Target Encoding:")
print(df_clean[['Route', 'Min Delay', 'Route_Encoded']].head())

After Target Encoding:
   Route  Min Delay  Route_Encoded
0     91         81      22.526583
1     69         22      16.595540
2     35         30      10.899409
3    900         17      15.481256
4     40          0      27.678139


In [ ]:
df_clean.drop(columns=['Route'],inplace=True)

In [ ]:
df_clean

,Time,Day,Incident,Min Delay,EGLINTON,FINCH,LAWRENCE,STEELES,JANE,SHEPPARD,...,LUMSDEN,DUNDAS WEST,WESTHUMBER,WELLESL,INDUSTRY,WEATHERELL,MORNINGSID,AUKLAND,MURRAY ROSS,Route_Encoded
0,02:30,Sunday,Diversion,81,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,22.526583
1,02:34,Sunday,Security,22,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,16.595540
2,03:06,Sunday,Cleaning - Unsanitary,30,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10.899409
3,03:14,Sunday,Security,17,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,15.481256
4,03:47,Sunday,Emergency Services,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,27.678139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39144,01:26,Sunday,Operations - Operator,28,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,13.684122
39145,01:31,Sunday,Emergency Services,10,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,20.659361
39146,01:40,Sunday,Diversion,33,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,13.389483
39147,01:54,Sunday,Emergency Services,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,18.361671


In [ ]:
# Convert to datetime and extract components
df_clean['Time'] = pd.to_datetime(df_clean['Time'], format='%H:%M')
df_clean['Hour'] = df_clean['Time'].dt.hour
df_clean['Minute'] = df_clean['Time'].dt.minute

# Cyclic encoding (preserves time continuity)
df_clean['Hour_sin'] = np.sin(2 * np.pi * df_clean['Hour']/24)
df_clean['Hour_cos'] = np.cos(2 * np.pi * df_clean['Hour']/24)
df_clean['Minute_sin'] = np.sin(2 * np.pi * df_clean['Minute']/60)
df_clean['Minute_cos'] = np.cos(2 * np.pi * df_clean['Minute']/60)

# Drop original columns if needed
df_clean = df_clean.drop(['Time', 'Hour', 'Minute'], axis=1)

In [ ]:
df_clean

,Day,Incident,Min Delay,EGLINTON,FINCH,LAWRENCE,STEELES,JANE,SHEPPARD,DUFFERIN,...,INDUSTRY,WEATHERELL,MORNINGSID,AUKLAND,MURRAY ROSS,Route_Encoded,Hour_sin,Hour_cos,Minute_sin,Minute_cos
0,Sunday,Diversion,81,0,0,0,0,0,0,0,...,0,0,0,0,0,22.526583,0.500000,0.866025,5.665539e-16,-1.000000
1,Sunday,Security,22,0,0,0,0,0,0,0,...,0,0,0,0,0,16.595540,0.500000,0.866025,-4.067366e-01,-0.913545
2,Sunday,Cleaning - Unsanitary,30,0,0,0,0,0,0,0,...,0,0,0,0,0,10.899409,0.707107,0.707107,5.877853e-01,0.809017
3,Sunday,Security,17,0,0,0,0,0,0,0,...,0,0,0,0,0,15.481256,0.707107,0.707107,9.945219e-01,0.104528
4,Sunday,Emergency Services,0,0,0,0,0,0,0,0,...,0,0,0,0,0,27.678139,0.707107,0.707107,-9.781476e-01,0.207912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39144,Sunday,Operations - Operator,28,1,0,0,0,0,0,0,...,0,0,0,0,0,13.684122,0.258819,0.965926,4.067366e-01,-0.913545
39145,Sunday,Emergency Services,10,0,0,0,0,0,0,0,...,0,0,0,0,0,20.659361,0.258819,0.965926,-1.045285e-01,-0.994522
39146,Sunday,Diversion,33,0,0,0,0,0,0,0,...,0,0,0,0,0,13.389483,0.258819,0.965926,-8.660254e-01,-0.500000
39147,Sunday,Emergency Services,0,0,0,0,0,0,0,0,...,0,0,0,0,0,18.361671,0.258819,0.965926,-5.877853e-01,0.809017


In [ ]:
q25 = df_clean['Min Delay'].quantile(0.25)
q50 = df_clean['Min Delay'].quantile(0.50)
q75 = df_clean['Min Delay'].quantile(0.75)

# Define a function to label the data
def classify_delay(value):
    if value <= q25:
        return 0  # Very Low
    elif value <= q50:
        return 1  # Low
    elif value <= q75:
        return 2  # Moderate
    else:
        return 3  # High

# Apply the function
df_clean['Min Delay'] = df_clean['Min Delay'].apply(classify_delay)


In [ ]:
df_clean['Min Delay']

,Min Delay
0,3
1,3
2,3
3,2
4,0
...,...
39144,3
39145,1
39146,3
39147,0


In [ ]:
X = df_clean.drop(columns=['Min Delay'])
y = df_clean['Min Delay']

In [ ]:
X

,Day,Incident,EGLINTON,FINCH,LAWRENCE,STEELES,JANE,SHEPPARD,DUFFERIN,BATHURST,...,INDUSTRY,WEATHERELL,MORNINGSID,AUKLAND,MURRAY ROSS,Route_Encoded,Hour_sin,Hour_cos,Minute_sin,Minute_cos
0,Sunday,Diversion,0,0,0,0,0,0,0,0,...,0,0,0,0,0,22.526583,0.500000,0.866025,5.665539e-16,-1.000000
1,Sunday,Security,0,0,0,0,0,0,0,0,...,0,0,0,0,0,16.595540,0.500000,0.866025,-4.067366e-01,-0.913545
2,Sunday,Cleaning - Unsanitary,0,0,0,0,0,0,0,0,...,0,0,0,0,0,10.899409,0.707107,0.707107,5.877853e-01,0.809017
3,Sunday,Security,0,0,0,0,0,0,0,0,...,0,0,0,0,0,15.481256,0.707107,0.707107,9.945219e-01,0.104528
4,Sunday,Emergency Services,0,0,0,0,0,0,0,0,...,0,0,0,0,0,27.678139,0.707107,0.707107,-9.781476e-01,0.207912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39144,Sunday,Operations - Operator,1,0,0,0,0,0,0,0,...,0,0,0,0,0,13.684122,0.258819,0.965926,4.067366e-01,-0.913545
39145,Sunday,Emergency Services,0,0,0,0,0,0,0,0,...,0,0,0,0,0,20.659361,0.258819,0.965926,-1.045285e-01,-0.994522
39146,Sunday,Diversion,0,0,0,0,0,0,0,0,...,0,0,0,0,0,13.389483,0.258819,0.965926,-8.660254e-01,-0.500000
39147,Sunday,Emergency Services,0,0,0,0,0,0,0,0,...,0,0,0,0,0,18.361671,0.258819,0.965926,-5.877853e-01,0.809017


In [ ]:
X_encoded = pd.get_dummies(X, columns=['Day', 'Incident'])

In [ ]:
X_encoded

,EGLINTON,FINCH,LAWRENCE,STEELES,JANE,SHEPPARD,DUFFERIN,BATHURST,YONGE,WESTON,...,Incident_Emergency Services,Incident_General Delay,Incident_Held By,Incident_Investigation,Incident_Mechanical,Incident_Operations - Operator,Incident_Road Blocked - NON-TTC Collision,Incident_Security,Incident_Utilized Off Route,Incident_Vision
0,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
2,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
4,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39144,1,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
39145,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
39146,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
39147,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42  # For reproducibility
)


In [ ]:
X_train

,EGLINTON,FINCH,LAWRENCE,STEELES,JANE,SHEPPARD,DUFFERIN,BATHURST,YONGE,WESTON,...,Incident_Emergency Services,Incident_General Delay,Incident_Held By,Incident_Investigation,Incident_Mechanical,Incident_Operations - Operator,Incident_Road Blocked - NON-TTC Collision,Incident_Security,Incident_Utilized Off Route,Incident_Vision
2296,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
8406,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
24541,0,0,0,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
13834,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
2392,1,0,0,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6361,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
11438,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
38613,0,0,0,0,1,0,1,0,0,0,...,False,True,False,False,False,False,False,False,False,False
866,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False


In [ ]:
# Get list of float columns to scale
float_cols = X_train.select_dtypes(include=['float']).columns.tolist()

# Columns to leave unchanged (int/bool)
other_cols = X_train.select_dtypes(include=['int', 'bool']).columns.tolist()

In [ ]:
float_cols

['Route_Encoded', 'Hour_sin', 'Hour_cos', 'Minute_sin', 'Minute_cos']

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scale', StandardScaler(), float_cols),  # Only scale floats
        ('passthrough', 'passthrough', other_cols)  # Leave others unchanged
    ])

In [ ]:
# Apply scaling only to float columns
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)  # Use same scaler as train


In [ ]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=float_cols + other_cols  # Maintain original column order
)
X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=float_cols + other_cols
)

print("Scaled columns:", float_cols)
print("Unchanged columns:", other_cols)

Scaled columns: ['Route_Encoded', 'Hour_sin', 'Hour_cos', 'Minute_sin', 'Minute_cos']
Unchanged columns: ['EGLINTON', 'FINCH', 'LAWRENCE', 'STEELES', 'JANE', 'SHEPPARD', 'DUFFERIN', 'BATHURST', 'YONGE', 'WESTON', 'KENNEDY STATION', 'KEELE', 'WILSON', 'KIPLING STATION', 'WILSON STATION', 'FINCH STATION', 'EGLINTON STATION', 'MARKHAM', 'PIONEER VILLAGE STATIO', 'ST CLAIR', 'ELLESMERE', 'KENNEDY', 'WARDEN', 'MCCOWAN', 'BLOOR', 'KIPLING', 'PAPE STATION', 'WARDEN STATION', 'MIDLAND', 'MORNINGSIDE', 'DON MILLS', 'SCARBOROUGH CENTRE STA', 'DUNDAS', 'QUEEN', 'DANFORTH', 'KINGSTON', 'EGLINTON WEST STATION', 'ISLINGTON', 'VICTORIA PARK', 'SHEPPARD WEST STATION', 'VICTORIA PARK STATION', 'FINCH WEST STATION', 'DON MILLS STATION', 'JANE STATION', 'KEELE STATION', 'YORK MILLS STATION', 'ALBION', 'LANSDOWNE', 'BRIMLEY', 'YORK MILLS', 'BAY', 'BAYVIEW', 'LAWRENCE WEST STATION', 'KING', 'SHERBOURNE', 'BIRCHMOUNT', 'ISLINGTON STATION', 'DUPONT', 'BATHURST STATION', 'MARTIN GROVE', 'MCNICOLL', 'LESLIE', 

In [ ]:
X_train_scaled

,Route_Encoded,Hour_sin,Hour_cos,Minute_sin,Minute_cos,EGLINTON,FINCH,LAWRENCE,STEELES,JANE,...,Incident_Emergency Services,Incident_General Delay,Incident_Held By,Incident_Investigation,Incident_Mechanical,Incident_Operations - Operator,Incident_Road Blocked - NON-TTC Collision,Incident_Security,Incident_Utilized Off Route,Incident_Vision
0,0.872813,-1.11771,0.784704,0.722138,1.237394,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
1,-0.717991,-0.980971,-0.429813,0.015636,1.427061,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
2,0.738449,-0.479971,1.756559,-0.929849,1.063433,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
3,-0.492869,1.172264,-0.761294,-1.034432,0.958652,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
4,-0.427043,0.888786,1.756559,-1.034432,-0.935912,1,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30947,3.256867,0.888786,1.756559,-0.814908,-1.133948,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
30948,0.486715,-0.479971,-1.015649,1.239334,0.719215,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
30949,-1.007248,-1.164349,0.370455,0.452278,1.357772,0,0,0,0,1,...,False,True,False,False,False,False,False,False,False,False
30950,-0.099637,0.558668,1.916453,-0.814908,-1.133948,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False


In [ ]:
y_train

,Min Delay
2296,2
8406,0
24541,2
13834,2
2392,3
...,...
6361,0
11438,3
38613,0
866,2


In [ ]:
# Define your float columns (as provided)
float_cols = ['Route_Encoded', 'Hour_sin', 'Hour_cos', 'Minute_sin', 'Minute_cos']

# 1. Convert float columns
for col in float_cols:
    X_train_scaled[col] = X_train_scaled[col].astype(float)
    X_test_scaled[col] = X_test_scaled[col].astype(float)

# 2. Convert remaining columns to int (excluding float_cols)
other_cols = [col for col in X_train_scaled.columns if col not in float_cols]
for col in other_cols:
    X_train_scaled[col] = X_train_scaled[col].astype(int)
    X_test_scaled[col] = X_test_scaled[col].astype(int)

# Verify dtypes
print("Train dtypes:")
print(X_train_scaled.dtypes)

print("\nTest dtypes:")
print(X_test_scaled.dtypes)

Train dtypes:
Route_Encoded                                float64
Hour_sin                                     float64
Hour_cos                                     float64
Minute_sin                                   float64
Minute_cos                                   float64
                                              ...   
Incident_Operations - Operator                 int64
Incident_Road Blocked - NON-TTC Collision      int64
Incident_Security                              int64
Incident_Utilized Off Route                    int64
Incident_Vision                                int64
Length: 452, dtype: object

Test dtypes:
Route_Encoded                                float64
Hour_sin                                     float64
Hour_cos                                     float64
Minute_sin                                   float64
Minute_cos                                   float64
                                              ...   
Incident_Operations - Operator              

In [ ]:
# Ensure y_train and y_test are integers
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Define classification models
models = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss'),
    "Random Forest": RandomForestClassifier(random_state=42),
    "LinearSVM": CalibratedClassifierCV(
        LinearSVC(random_state=42, max_iter=10000, dual=False),
        method='sigmoid',
        cv=3
    )
}

# Train, evaluate, and save each model
results = {}
for name, model in models.items():
    print(f"\nTraining {name}...")

    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None

    # Evaluate
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "ROC AUC (OvO)": roc_auc_score(y_test, y_proba, multi_class='ovo') if y_proba is not None else "N/A",
        "ROC AUC (OvR)": roc_auc_score(y_test, y_proba, multi_class='ovr') if y_proba is not None else "N/A"
    }

    # Save model
    model_path = f"{name.replace(' ', '_').lower()}_classifier.pkl"
    joblib.dump(model, model_path)
    print(f"Saved {name} to {model_path}")

    # Confusion matrix
    print(f"\nConfusion Matrix ({name}):")
    print(confusion_matrix(y_test, y_pred))

# Convert results to DataFrame
results_df = pd.DataFrame(results).T
print("\nModel Performance Comparison:")
print(results_df)


Training XGBoost...
Saved XGBoost to xgboost_classifier.pkl

Confusion Matrix (XGBoost):
[[1445  264  541  137]
 [ 366  691  475   38]
 [ 330  204 1677  268]
 [ 102   28  383  790]]

Training Random Forest...
Saved Random Forest to random_forest_classifier.pkl

Confusion Matrix (Random Forest):
[[1516  257  488  126]
 [ 421  679  428   42]
 [ 403  179 1645  252]
 [ 125   42  370  766]]

Training LinearSVM...
Saved LinearSVM to linearsvm_classifier.pkl

Confusion Matrix (LinearSVM):
[[1373  266  627  121]
 [ 506  486  546   32]
 [ 470  201 1592  216]
 [ 158   50  442  653]]

Model Performance Comparison:
               Accuracy  Precision    Recall        F1  ROC AUC (OvO)  \
XGBoost        0.594780   0.599315  0.594780  0.592520       0.827096   
Random Forest  0.595167   0.597272  0.595167  0.592199       0.826254   
LinearSVM      0.530301   0.533812  0.530301  0.523650       0.779378   

               ROC AUC (OvR)  
XGBoost             0.822014  
Random Forest       0.822264  
Li

In [ ]:
# Ensure y_train and y_test are integers
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Define models with limited hyperparameter grids
models = {
    "XGBoost": {
        "model": XGBClassifier(random_state=42, eval_metric='mlogloss'),
        "params": {
            'n_estimators': [50, 100],
            'max_depth': [3, 5],
            'learning_rate': [0.1, 0.3]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            'n_estimators': [50, 100],
            'max_depth': [None, 5],
            'min_samples_split': [2, 5]
        }
    },
    "LinearSVM": {
        "model": CalibratedClassifierCV(
            LinearSVC(random_state=42, max_iter=10000, dual=False),
            method='sigmoid',
            cv=3
        ),
        "params": {
            'estimator__C': [0.1, 1, 10],  # Corrected parameter name
            'estimator__loss': ['squared_hinge']
        }
    }
}

# Train/evaluate with grid search
results = {}
for name, config in models.items():
    print(f"\n=== Optimizing {name} ===")
    start_time = time()

    search = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train_scaled, y_train)
    best_model = search.best_estimator_

    # Predict and evaluate
    y_pred = best_model.predict(X_test_scaled)
    y_proba = best_model.predict_proba(X_test_scaled) if hasattr(best_model, "predict_proba") else None

    results[name] = {
        "Best Params": search.best_params_,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "ROC AUC (OvO)": roc_auc_score(y_test, y_proba, multi_class='ovo') if y_proba is not None else "N/A",
        "ROC AUC (OvR)": roc_auc_score(y_test, y_proba, multi_class='ovr') if y_proba is not None else "N/A",
        "Training Time (min)": round((time() - start_time)/60, 2)
    }

    joblib.dump(best_model, f"{name.lower()}_best_classifier.pkl")
    print(f"Confusion Matrix ({name}):\n{confusion_matrix(y_test, y_pred)}")

# Display results
results_df = pd.DataFrame(results).T
print("\n=== Final Results ===")
print(results_df[['Best Params', 'Accuracy', 'F1', 'ROC AUC (OvO)', 'Training Time (min)']])


=== Optimizing XGBoost ===
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Confusion Matrix (XGBoost):
[[1410  256  579  142]
 [ 386  651  501   32]
 [ 335  186 1694  264]
 [ 107   27  393  776]]

=== Optimizing Random Forest ===
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Confusion Matrix (Random Forest):
[[1501  236  516  134]
 [ 409  659  466   36]
 [ 372  182 1673  252]
 [ 130   37  379  757]]

=== Optimizing LinearSVM ===
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Confusion Matrix (LinearSVM):
[[1375  278  614  120]
 [ 504  499  536   31]
 [ 474  209 1578  218]
 [ 156   48  442  657]]

=== Final Results ===
                                                     Best Params  Accuracy  \
XGBoost        {'learning_rate': 0.3, 'max_depth': 5, 'n_esti...  0.585476   
Random Forest  {'max_depth': None, 'min_samples_split': 5, 'n...    0.5931   
LinearSVM      {'estimator__C': 0.1, 'estimator__loss': 'squa...  0.530947   

                     F

In [ ]:
results_df

,Best Params,Accuracy,Precision,Recall,F1,ROC AUC (OvO),ROC AUC (OvR),Training Time (min)
XGBoost,"{'learning_rate': 0.3, 'max_depth': 5, 'n_esti...",0.585476,0.591203,0.585476,0.58229,0.822926,0.817695,2.32
Random Forest,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.5931,0.596689,0.5931,0.589681,0.829006,0.825011,2.51
LinearSVM,"{'estimator__C': 0.1, 'estimator__loss': 'squa...",0.530947,0.534205,0.530947,0.524947,0.781259,0.775203,0.57


In [ ]:
# Load the model using joblib
with open('random forest_best_classifier.pkl', 'rb') as file:
    rf_model = joblib.load(file)

y_pred = rf_model.predict(X_test_scaled)

In [ ]:
y_pred[:10] # Displaying first 10 predicted values

array([2, 1, 0, 2, 3, 0, 1, 2, 0, 2])

In [ ]:
y_test[:10] # Displaying first 10 actual values

,Min Delay
18196,2
12007,0
27856,0
37805,0
1683,3
7417,1
24972,1
18197,2
16496,1
9532,1


In [2]:
# Load data for 2024
df = pd.read_excel('ttc-bus-delay-data-2024.xlsx')

In [3]:
df.head()

,Route,Time,Day,Location,Incident,Min Delay
0,89,02:08,Monday,KEELE AND GLENLAKE,Vision,10
1,39,02:30,Monday,FINCH STATION,General Delay,20
2,300,03:13,Monday,BLOOR AND MANNING,General Delay,0
3,65,03:23,Monday,PARLIAMENT AND BLOOR,Security,0
4,113,03:37,Monday,MAIN STATION,Security,0


In [4]:
df.describe(include='all')

,Route,Time,Day,Location,Incident,Min Delay
count,59021.0,59643,59643,59643,59643,59643.000000
unique,264.0,1440,7,10505,12,NaN
top,32.0,16:00,Friday,KENNEDY STATION,Mechanical,NaN
freq,1786.0,152,9593,1798,19883,NaN
mean,NaN,NaN,NaN,NaN,NaN,21.218433
std,NaN,NaN,NaN,NaN,NaN,53.808012
min,NaN,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,NaN,8.000000
50%,NaN,NaN,NaN,NaN,NaN,11.000000
75%,NaN,NaN,NaN,NaN,NaN,20.000000


In [5]:
df.dtypes

,0
Route,object
Time,object
Day,object
Location,object
Incident,object
Min Delay,int64


In [6]:
# 1. Clean and standardize location data
def clean_location(loc):
    # Replace common intersection indicators with standardized ' AND '
    loc = re.sub(r'(&|\bat\b)', ' AND ', str(loc), flags=re.IGNORECASE)
    # Remove special characters (including dashes), numbers, and extra spaces
    loc = re.sub(r'[^a-zA-Z\s]', '', loc)  # Removes dashes and other special chars
    # Collapse multiple ANDs and spaces
    loc = re.sub(r'\s+AND\s+', ' AND ', loc, flags=re.IGNORECASE)
    loc = re.sub(r'\s+', ' ', loc).strip().upper()
    return loc

df['Location'] = df['Location'].apply(clean_location)

# 2. Split locations into components (handling AND, &, and at)
df['Location_Components'] = df['Location'].str.split(' AND ')

# 3. Calculate component frequencies
all_components = df['Location_Components'].explode()
component_counts = all_components.value_counts()

# 4. Get components with >20 occurrences
frequent_components = component_counts[component_counts > 20].index.tolist()

# 5. Filter rows where all components are frequent
mask = df['Location_Components'].apply(
    lambda x: all(comp in frequent_components for comp in x)
)
df_filtered = df[mask].copy()

# 6. Create binary encoding for frequent components
if frequent_components:
    mlb = MultiLabelBinarizer(classes=frequent_components)
    encoded_locations = mlb.fit_transform(df_filtered['Location_Components'])

    location_df = pd.DataFrame(encoded_locations,
                             columns=mlb.classes_,
                             index=df_filtered.index)
    df_encoded = pd.concat([df_filtered, location_df], axis=1)

    # Cleanup columns
    df_encoded = df_encoded.drop(columns=['Location_Components'])
else:
    print("Warning: No locations met the frequency threshold!")
    df_encoded = df_filtered

# 7. (Optional) Reset index if needed
df_encoded = df_encoded.reset_index(drop=True)

# Show results
print(f"Original rows: {len(df)}")
print(f"Kept rows after filtering: {len(df_encoded)}")
print(f"Total location components found: {len(component_counts)}")
print(f"Frequent components (>20 occurrences): {len(frequent_components)}")
print("\nSample frequent components:", frequent_components[:10])

Original rows: 59643
Kept rows after filtering: 42280
Total location components found: 7495
Frequent components (>20 occurrences): 466

Sample frequent components: ['EGLINTON', 'LAWRENCE', 'FINCH', 'STEELES', 'JANE', 'SHEPPARD', 'DUFFERIN', 'KENNEDY STATION', 'YONGE', 'BATHURST']


In [7]:
df_encoded.head()

,Route,Time,Day,Location,Incident,Min Delay,EGLINTON,LAWRENCE,FINCH,STEELES,...,SAUL,QUEENS PARK,GULLIVER,HIGHVIEW,AUKLAND,HIGH PARK,JONES,THE WEST,BOMBAY LOOP,SELECT
0,89,02:08,Monday,KEELE AND GLENLAKE,Vision,10,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,39,02:30,Monday,FINCH STATION,General Delay,20,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,65,03:23,Monday,PARLIAMENT AND BLOOR,Security,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,113,03:37,Monday,MAIN STATION,Security,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,171,04:00,Monday,MOUNT DENNIS GARAGE,General Delay,20,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
df_encoded.isnull().sum()

,0
Route,331
Time,0
Day,0
Location,0
Incident,0
...,...
HIGH PARK,0
JONES,0
THE WEST,0
BOMBAY LOOP,0


In [9]:
df_encoded.dropna(inplace=True)

In [10]:
df_encoded.isnull().sum()

,0
Route,0
Time,0
Day,0
Location,0
Incident,0
...,...
HIGH PARK,0
JONES,0
THE WEST,0
BOMBAY LOOP,0


In [11]:
df_encoded['Route'].value_counts()

,count
Route,
32,1371
52,1159
36,1155
29,1019
35,1014
...,...
340,1
00RAD,1
601,1


In [12]:
#Robust text filtering
def is_numeric_route(route):
    try:
        # Handle NaN/None, strings, and numeric types
        return pd.notna(route) and str(route).isdigit()
    except:
        return False

# Apply filtering
mask = df_encoded['Route'].apply(is_numeric_route)
df_clean = df_encoded[mask].copy()

# Convert to integers (now safe)
df_clean['Route'] = df_clean['Route'].astype(int)

print(f"Original rows: {len(df_encoded)}")
print(f"Clean rows: {len(df_clean)}")
print("\nCleaned data:")
print(df_clean)

Original rows: 41949
Clean rows: 41869

Cleaned data:
       Route   Time      Day                Location               Incident  \
0         89  02:08   Monday      KEELE AND GLENLAKE                 Vision   
1         39  02:30   Monday           FINCH STATION          General Delay   
2         65  03:23   Monday    PARLIAMENT AND BLOOR               Security   
3        113  03:37   Monday            MAIN STATION               Security   
4        171  04:00   Monday     MOUNT DENNIS GARAGE          General Delay   
...      ...    ...      ...                     ...                    ...   
42275     75  21:29  Tuesday      WOODBINE AND QUEEN               Security   
42276    129  22:06  Tuesday   MCCOWAN AND SANDHURST  Operations - Operator   
42277    102  23:15  Tuesday  MORNINGSIDE AND PASSMO             Mechanical   
42278     95  00:44  Tuesday  ELLESMERE AND BIRCHMOU     Emergency Services   
42279     65  01:13  Tuesday    CASTLE FRANK STATION               Security  

In [13]:
df_clean['Route'].value_counts()

,count
Route,
32,1371
52,1159
36,1155
29,1019
35,1014
...,...
176,1
1060,1
340,1


In [14]:
df_clean.drop(columns=['Location'],inplace=True)

In [15]:
df_clean

,Route,Time,Day,Incident,Min Delay,EGLINTON,LAWRENCE,FINCH,STEELES,JANE,...,SAUL,QUEENS PARK,GULLIVER,HIGHVIEW,AUKLAND,HIGH PARK,JONES,THE WEST,BOMBAY LOOP,SELECT
0,89,02:08,Monday,Vision,10,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,39,02:30,Monday,General Delay,20,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,65,03:23,Monday,Security,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,113,03:37,Monday,Security,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,171,04:00,Monday,General Delay,20,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42275,75,21:29,Tuesday,Security,10,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
42276,129,22:06,Tuesday,Operations - Operator,27,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
42277,102,23:15,Tuesday,Mechanical,20,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
42278,95,00:44,Tuesday,Emergency Services,17,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
# Initialize TargetEncoder with smoothing to prevent overfitting
encoder = TargetEncoder(smooth="auto", target_type='continuous')

# Fit and transform the Route column
df_clean['Route_Encoded'] = encoder.fit_transform(df_clean[['Route']], df_clean['Min Delay'])

print("After Target Encoding:")
print(df_clean[['Route', 'Min Delay', 'Route_Encoded']].head())

After Target Encoding:
   Route  Min Delay  Route_Encoded
0     89         10      12.847537
1     39         20      17.749242
2     65          0      19.373716
3    113          0      20.670546
4    171         20      37.941849


In [17]:
df_clean.drop(columns=['Route'],inplace=True)

In [18]:
df_clean

,Time,Day,Incident,Min Delay,EGLINTON,LAWRENCE,FINCH,STEELES,JANE,SHEPPARD,...,QUEENS PARK,GULLIVER,HIGHVIEW,AUKLAND,HIGH PARK,JONES,THE WEST,BOMBAY LOOP,SELECT,Route_Encoded
0,02:08,Monday,Vision,10,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,12.847537
1,02:30,Monday,General Delay,20,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,17.749242
2,03:23,Monday,Security,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,19.373716
3,03:37,Monday,Security,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,20.670546
4,04:00,Monday,General Delay,20,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,37.941849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42275,21:29,Tuesday,Security,10,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,18.077921
42276,22:06,Tuesday,Operations - Operator,27,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,17.071844
42277,23:15,Tuesday,Mechanical,20,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,21.580687
42278,00:44,Tuesday,Emergency Services,17,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,15.526604


In [19]:
# Convert to datetime and extract components
df_clean['Time'] = pd.to_datetime(df_clean['Time'], format='%H:%M')
df_clean['Hour'] = df_clean['Time'].dt.hour
df_clean['Minute'] = df_clean['Time'].dt.minute

# Cyclic encoding (preserves time continuity)
df_clean['Hour_sin'] = np.sin(2 * np.pi * df_clean['Hour']/24)
df_clean['Hour_cos'] = np.cos(2 * np.pi * df_clean['Hour']/24)
df_clean['Minute_sin'] = np.sin(2 * np.pi * df_clean['Minute']/60)
df_clean['Minute_cos'] = np.cos(2 * np.pi * df_clean['Minute']/60)

# Drop original columns if needed
df_clean = df_clean.drop(['Time', 'Hour', 'Minute'], axis=1)

In [20]:
df_clean

,Day,Incident,Min Delay,EGLINTON,LAWRENCE,FINCH,STEELES,JANE,SHEPPARD,DUFFERIN,...,HIGH PARK,JONES,THE WEST,BOMBAY LOOP,SELECT,Route_Encoded,Hour_sin,Hour_cos,Minute_sin,Minute_cos
0,Monday,Vision,10,0,0,0,0,0,0,0,...,0,0,0,0,0,12.847537,0.500000,0.866025,7.431448e-01,6.691306e-01
1,Monday,General Delay,20,0,0,0,0,0,0,0,...,0,0,0,0,0,17.749242,0.500000,0.866025,5.665539e-16,-1.000000e+00
2,Monday,Security,0,0,0,0,0,0,0,0,...,0,0,0,0,0,19.373716,0.707107,0.707107,6.691306e-01,-7.431448e-01
3,Monday,Security,0,0,0,0,0,0,0,0,...,0,0,0,0,0,20.670546,0.707107,0.707107,-6.691306e-01,-7.431448e-01
4,Monday,General Delay,20,0,0,0,0,0,0,0,...,0,0,0,0,0,37.941849,0.866025,0.500000,0.000000e+00,1.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42275,Tuesday,Security,10,0,0,0,0,0,0,0,...,0,0,0,0,0,18.077921,-0.707107,0.707107,1.045285e-01,-9.945219e-01
42276,Tuesday,Operations - Operator,27,0,0,0,0,0,0,0,...,0,0,0,0,0,17.071844,-0.500000,0.866025,5.877853e-01,8.090170e-01
42277,Tuesday,Mechanical,20,0,0,0,0,0,0,0,...,0,0,0,0,0,21.580687,-0.258819,0.965926,1.000000e+00,2.832769e-16
42278,Tuesday,Emergency Services,17,0,0,0,0,0,0,0,...,0,0,0,0,0,15.526604,0.000000,1.000000,-9.945219e-01,-1.045285e-01


In [21]:
q25 = df_clean['Min Delay'].quantile(0.25)
q50 = df_clean['Min Delay'].quantile(0.50)
q75 = df_clean['Min Delay'].quantile(0.75)

# Step 2: Define a function to label the data
def classify_delay(value):
    if value <= q25:
        return 0  # Very Low
    elif value <= q50:
        return 1  # Low
    elif value <= q75:
        return 2  # Moderate
    else:
        return 3  # High

# Step 3: Apply the function
df_clean['Min Delay'] = df_clean['Min Delay'].apply(classify_delay)


In [22]:
df_clean['Min Delay']

,Min Delay
0,1
1,2
2,0
3,0
4,2
...,...
42275,1
42276,3
42277,2
42278,2


In [23]:
X = df_clean.drop(columns=['Min Delay'])
y = df_clean['Min Delay']

In [24]:
X

,Day,Incident,EGLINTON,LAWRENCE,FINCH,STEELES,JANE,SHEPPARD,DUFFERIN,KENNEDY STATION,...,HIGH PARK,JONES,THE WEST,BOMBAY LOOP,SELECT,Route_Encoded,Hour_sin,Hour_cos,Minute_sin,Minute_cos
0,Monday,Vision,0,0,0,0,0,0,0,0,...,0,0,0,0,0,12.847537,0.500000,0.866025,7.431448e-01,6.691306e-01
1,Monday,General Delay,0,0,0,0,0,0,0,0,...,0,0,0,0,0,17.749242,0.500000,0.866025,5.665539e-16,-1.000000e+00
2,Monday,Security,0,0,0,0,0,0,0,0,...,0,0,0,0,0,19.373716,0.707107,0.707107,6.691306e-01,-7.431448e-01
3,Monday,Security,0,0,0,0,0,0,0,0,...,0,0,0,0,0,20.670546,0.707107,0.707107,-6.691306e-01,-7.431448e-01
4,Monday,General Delay,0,0,0,0,0,0,0,0,...,0,0,0,0,0,37.941849,0.866025,0.500000,0.000000e+00,1.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42275,Tuesday,Security,0,0,0,0,0,0,0,0,...,0,0,0,0,0,18.077921,-0.707107,0.707107,1.045285e-01,-9.945219e-01
42276,Tuesday,Operations - Operator,0,0,0,0,0,0,0,0,...,0,0,0,0,0,17.071844,-0.500000,0.866025,5.877853e-01,8.090170e-01
42277,Tuesday,Mechanical,0,0,0,0,0,0,0,0,...,0,0,0,0,0,21.580687,-0.258819,0.965926,1.000000e+00,2.832769e-16
42278,Tuesday,Emergency Services,0,0,0,0,0,0,0,0,...,0,0,0,0,0,15.526604,0.000000,1.000000,-9.945219e-01,-1.045285e-01


In [25]:
X_encoded = pd.get_dummies(X, columns=['Day', 'Incident'])

In [26]:
X_encoded

,EGLINTON,LAWRENCE,FINCH,STEELES,JANE,SHEPPARD,DUFFERIN,KENNEDY STATION,YONGE,BATHURST,...,Incident_Diversion,Incident_Emergency Services,Incident_General Delay,Incident_Investigation,Incident_Mechanical,Incident_Operations - Operator,Incident_Road Blocked - NON-TTC Collision,Incident_Security,Incident_Utilized Off Route,Incident_Vision
0,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,True
1,0,0,0,0,0,0,0,0,0,0,...,False,False,True,False,False,False,False,False,False,False
2,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
3,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
4,0,0,0,0,0,0,0,0,0,0,...,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42275,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
42276,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
42277,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
42278,0,0,0,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42  # For reproducibility
)


In [28]:
X_train

,EGLINTON,LAWRENCE,FINCH,STEELES,JANE,SHEPPARD,DUFFERIN,KENNEDY STATION,YONGE,BATHURST,...,Incident_Diversion,Incident_Emergency Services,Incident_General Delay,Incident_Investigation,Incident_Mechanical,Incident_Operations - Operator,Incident_Road Blocked - NON-TTC Collision,Incident_Security,Incident_Utilized Off Route,Incident_Vision
12951,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
40167,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
26903,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
6056,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
16559,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6320,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
11388,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
38511,0,0,0,0,0,0,0,0,1,0,...,False,False,False,False,True,False,False,False,False,False
867,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False


In [29]:
# Get list of float columns to scale
float_cols = X_train.select_dtypes(include=['float']).columns.tolist()

# Columns to leave unchanged (int/bool)
other_cols = X_train.select_dtypes(include=['int', 'bool']).columns.tolist()

In [30]:
float_cols

['Route_Encoded', 'Hour_sin', 'Hour_cos', 'Minute_sin', 'Minute_cos']

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        ('scale', StandardScaler(), float_cols),  # Only scale floats
        ('passthrough', 'passthrough', other_cols)  # Leave others unchanged
    ])

In [32]:
# Apply scaling only to float columns
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)  # Use same scaler as train


In [33]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=float_cols + other_cols  # Maintain original column order
)
X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=float_cols + other_cols
)

print("Scaled columns:", float_cols)
print("Unchanged columns:", other_cols)

Scaled columns: ['Route_Encoded', 'Hour_sin', 'Hour_cos', 'Minute_sin', 'Minute_cos']
Unchanged columns: ['EGLINTON', 'LAWRENCE', 'FINCH', 'STEELES', 'JANE', 'SHEPPARD', 'DUFFERIN', 'KENNEDY STATION', 'YONGE', 'BATHURST', 'WESTON', 'WILSON', 'KEELE', 'KIPLING STATION', 'ELLESMERE', 'KENNEDY', 'MARKHAM', 'EGLINTON STATION', 'ST CLAIR', 'WILSON STATION', 'FINCH STATION', 'BLOOR', 'WARDEN', 'PIONEER VILLAGE STATIO', 'KIPLING', 'MORNINGSIDE', 'MIDLAND', 'DUNDAS', 'MCCOWAN', 'DON MILLS', 'ISLINGTON', 'KINGSTON', 'WARDEN STATION', 'DANFORTH', 'SCARBOROUGH CENTRE STA', 'VICTORIA PARK', 'QUEEN', 'YORK MILLS STATION', 'EGLINTON WEST STATION', 'DON MILLS STATION', 'BAY', 'BRIMLEY', 'FINCH WEST STATION', 'KEELE STATION', 'YORK MILLS', 'BAYVIEW', 'SHEPPARD WEST STATION', 'BROADVIEW STATION', 'ALBION', 'VICTORIA PARK STATION', 'LANSDOWNE', 'KING', 'OVERLEA', 'AVENUE', 'PROGRESS', 'MAIN STREET STATION', 'FRONT', 'VICTORIA', 'BIRCHMOUNT', 'LESLIE', 'ISLINGTON STATION', 'MCNICOLL', 'JANE STATION', 'SH

In [34]:
X_train_scaled

,Route_Encoded,Hour_sin,Hour_cos,Minute_sin,Minute_cos,EGLINTON,LAWRENCE,FINCH,STEELES,JANE,...,Incident_Diversion,Incident_Emergency Services,Incident_General Delay,Incident_Investigation,Incident_Mechanical,Incident_Operations - Operator,Incident_Road Blocked - NON-TTC Collision,Incident_Security,Incident_Utilized Off Route,Incident_Vision
0,1.366897,0.575496,-1.17722,1.240072,0.703188,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
1,-0.499686,-0.974291,-0.438873,1.36083,-0.436239,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
2,0.138321,1.596679,0.353471,1.36083,0.434205,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
3,0.24294,0.90779,-1.018909,0.305442,-1.378648,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
4,-0.746158,0.90779,-1.018909,-0.428683,1.338459,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33490,0.269134,-0.974291,-0.438873,1.240072,0.703188,0,0,0,0,0,...,False,False,False,False,False,True,False,False,False,False
33491,1.049889,1.412092,1.145814,-0.82458,1.13841,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
33492,1.765395,0.575496,1.884161,1.307558,0.571835,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
33493,0.766821,-0.469989,-1.018909,-0.428683,1.338459,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False


In [35]:
y_train

,Min Delay
12951,3
40167,2
26903,3
6056,0
16559,2
...,...
6320,2
11388,0
38511,3
867,3


In [36]:
# Define your float columns (as provided)
float_cols = ['Route_Encoded', 'Hour_sin', 'Hour_cos', 'Minute_sin', 'Minute_cos']

# 1. Convert float columns
for col in float_cols:
    X_train_scaled[col] = X_train_scaled[col].astype(float)
    X_test_scaled[col] = X_test_scaled[col].astype(float)

# 2. Convert remaining columns to int (excluding float_cols)
other_cols = [col for col in X_train_scaled.columns if col not in float_cols]
for col in other_cols:
    X_train_scaled[col] = X_train_scaled[col].astype(int)
    X_test_scaled[col] = X_test_scaled[col].astype(int)

# Verify dtypes
print("Train dtypes:")
print(X_train_scaled.dtypes)

print("\nTest dtypes:")
print(X_test_scaled.dtypes)

Train dtypes:
Route_Encoded                                float64
Hour_sin                                     float64
Hour_cos                                     float64
Minute_sin                                   float64
Minute_cos                                   float64
                                              ...   
Incident_Operations - Operator                 int64
Incident_Road Blocked - NON-TTC Collision      int64
Incident_Security                              int64
Incident_Utilized Off Route                    int64
Incident_Vision                                int64
Length: 490, dtype: object

Test dtypes:
Route_Encoded                                float64
Hour_sin                                     float64
Hour_cos                                     float64
Minute_sin                                   float64
Minute_cos                                   float64
                                              ...   
Incident_Operations - Operator              

In [37]:
# Ensure y_train and y_test are integers
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Define classification models
models = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss'),
    "Random Forest": RandomForestClassifier(random_state=42),
    "LinearSVM": CalibratedClassifierCV(
        LinearSVC(random_state=42, max_iter=10000, dual=False),
        method='sigmoid',
        cv=3
    )
}

# Train, evaluate, and save each model
results = {}
for name, model in models.items():
    print(f"\nTraining {name}...")

    # Train
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None

    # Evaluate
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "ROC AUC (OvO)": roc_auc_score(y_test, y_proba, multi_class='ovo') if y_proba is not None else "N/A",
        "ROC AUC (OvR)": roc_auc_score(y_test, y_proba, multi_class='ovr') if y_proba is not None else "N/A"
    }

    # Save model
    model_path = f"{name.replace(' ', '_').lower()}_classifier.pkl"
    joblib.dump(model, model_path)
    print(f"Saved {name} to {model_path}")

    # Confusion matrix
    print(f"\nConfusion Matrix ({name}):")
    print(confusion_matrix(y_test, y_pred))

# Convert results to DataFrame
results_df = pd.DataFrame(results).T
print("\nModel Performance Comparison:")
print(results_df)


Training XGBoost...
Saved XGBoost to xgboost_classifier.pkl

Confusion Matrix (XGBoost):
[[1261  380  469  142]
 [ 328 1005  659   78]
 [ 248  346 1756  217]
 [  99   66  467  853]]

Training Random Forest...
Saved Random Forest to random_forest_classifier.pkl

Confusion Matrix (Random Forest):
[[1373  354  404  121]
 [ 361 1097  553   59]
 [ 312  368 1705  182]
 [ 149   87  463  786]]

Training LinearSVM...
Saved LinearSVM to linearsvm_classifier.pkl

Confusion Matrix (LinearSVM):
[[1178  396  560  118]
 [ 441  769  805   55]
 [ 396  374 1648  149]
 [ 188   96  563  638]]

Model Performance Comparison:
               Accuracy  Precision    Recall        F1  ROC AUC (OvO)  \
XGBoost        0.582159   0.591308  0.582159  0.581372       0.823906   
Random Forest  0.592429   0.599156  0.592429  0.591987       0.825331   
LinearSVM      0.505493   0.519191  0.505493  0.501857       0.766717   

               ROC AUC (OvR)  
XGBoost             0.819449  
Random Forest       0.822354  
Li

In [38]:
# Ensure y_train and y_test are integers
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Define models with limited hyperparameter grids
models = {
    "XGBoost": {
        "model": XGBClassifier(random_state=42, eval_metric='mlogloss'),
        "params": {
            'n_estimators': [50, 100],
            'max_depth': [3, 5],
            'learning_rate': [0.1, 0.3]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            'n_estimators': [50, 100],
            'max_depth': [None, 5],
            'min_samples_split': [2, 5]
        }
    },
    "LinearSVM": {
        "model": CalibratedClassifierCV(
            LinearSVC(random_state=42, max_iter=10000, dual=False),
            method='sigmoid',
            cv=3
        ),
        "params": {
            'estimator__C': [0.1, 1, 10],
            'estimator__loss': ['squared_hinge']
        }
    }
}

# Train/evaluate with grid search
results = {}
for name, config in models.items():
    print(f"\n=== Optimizing {name} ===")
    start_time = time()

    search = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=3,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train_scaled, y_train)
    best_model = search.best_estimator_

    # Predict and evaluate
    y_pred = best_model.predict(X_test_scaled)
    y_proba = best_model.predict_proba(X_test_scaled) if hasattr(best_model, "predict_proba") else None

    results[name] = {
        "Best Params": search.best_params_,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1": f1_score(y_test, y_pred, average='weighted'),
        "ROC AUC (OvO)": roc_auc_score(y_test, y_proba, multi_class='ovo') if y_proba is not None else "N/A",
        "ROC AUC (OvR)": roc_auc_score(y_test, y_proba, multi_class='ovr') if y_proba is not None else "N/A",
        "Training Time (min)": round((time() - start_time)/60, 2)
    }

    joblib.dump(best_model, f"{name.lower()}_best_classifier.pkl")
    print(f"Confusion Matrix ({name}):\n{confusion_matrix(y_test, y_pred)}")

# Display results
results_df = pd.DataFrame(results).T
print("\n=== Final Results ===")
print(results_df[['Best Params', 'Accuracy', 'F1', 'ROC AUC (OvO)', 'Training Time (min)']])


=== Optimizing XGBoost ===
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Confusion Matrix (XGBoost):
[[1238  384  479  151]
 [ 318  996  680   76]
 [ 257  344 1757  209]
 [ 107   62  471  845]]

=== Optimizing Random Forest ===
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Confusion Matrix (Random Forest):
[[1372  337  428  115]
 [ 349 1091  572   58]
 [ 290  348 1750  179]
 [ 132   82  488  783]]

=== Optimizing LinearSVM ===
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Confusion Matrix (LinearSVM):
[[1182  399  556  115]
 [ 433  783  802   52]
 [ 397  377 1647  146]
 [ 190   94  572  629]]

=== Final Results ===
                                                     Best Params  Accuracy  \
XGBoost        {'learning_rate': 0.3, 'max_depth': 5, 'n_esti...  0.577502   
Random Forest  {'max_depth': None, 'min_samples_split': 5, 'n...  0.596609   
LinearSVM      {'estimator__C': 0.1, 'estimator__loss': 'squa...  0.506449   

                     F

In [39]:
results_df

,Best Params,Accuracy,Precision,Recall,F1,ROC AUC (OvO),ROC AUC (OvR),Training Time (min)
XGBoost,"{'learning_rate': 0.3, 'max_depth': 5, 'n_esti...",0.577502,0.587251,0.577502,0.576572,0.818358,0.813715,3.48
Random Forest,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.596609,0.605335,0.596609,0.596039,0.828879,0.825814,3.01
LinearSVM,"{'estimator__C': 0.1, 'estimator__loss': 'squa...",0.506449,0.521005,0.506449,0.50298,0.768514,0.764071,0.75


In [40]:
# Load the model using joblib to visualise first 5 predicted values to actual values
with open('random forest_best_classifier.pkl', 'rb') as file:
    rf_model = joblib.load(file)

y_pred = rf_model.predict(X_test_scaled)

In [41]:
y_pred[:10] #showing first 10 predicted values

array([2, 2, 0, 1, 2, 2, 1, 2, 2, 2])

In [42]:
y_test[:10] # displaying first actual 10 values

,Min Delay
40131,2
4592,3
28008,0
16197,1
16802,3
24350,2
25250,1
36053,3
15640,2
42124,1
